# 35 - ONE frozen-before LSA-R alternative: replay and full-chain check (NO TRAINING)

34 FAILED and stays FAILED. No threshold relaxation, anchor reroll, new pretrained model, optimizer,
training, NBIS run, new calibration identities, or validation/test images. This is the ONE authorized
alternative: use ALL cached original anchors of each query finger (up to24), not34's single anchor.
Reuse exact33 descriptors/anchors and authoritative thresholds. Source files are strictly read only.

For each before-input query/gallery pair, official LSA-R has a discrete assignment and selected ranks,
but continuous geometric relaxation is linear in local scores while those choices stay fixed.
Record `W = d LSA-R(S_before) / d S_before` with detached discrete assignment, then DETACH W:

`frozen_after = official_before + sum(W * (S_after - S_before))`.

The score-space Jacobian encodes the frozen assignment, rank selection and geometric propagation.
It is NOT a learned network, straight-through gradient through assignment, global differentiable
reproduction, novel optimization theory, or the recomputed matcher when matching/ranking changes.
Check before homogeneity, exact no-op, replay against ALL33 pairs and own finite differences at
predeclared calibration pairs. Report branch changes instead of hiding them or selecting another epsilon.

Before equality is crucial: thresholds have33's LSA-R scale, NOT34's local-cosine scale.
Any significant replay discrepancy blocks eligibility. Existing33 quality is development evidence
already inspected, not a new independent result. Geometry-only weight-mass AUC is reported: geometry
may explain discrimination, so a replay pass does NOT establish descriptor usefulness or restoration gains.

Six deterministic live cases: first byte-distinct calibration probe per DB, dry-like/moderate, all
same-DB nonduplicate calibration gallery identities. ORIGINAL23 seed20261110 selected best checkpoint
produces after patches; no checkpoint reselection. ALL anchors are sampled as aligned192px patches;
the same frozen DMD receives center128px. No predicted-image minutiae or geometry are used.
The after template is built from independently restored aligned patches, not necessarily descriptors
of one jointly restored native-orientation image. Later deployable inference and NBIS must be matched;
this technical check does not solve stitching, rotation/context differences or test-time anchor detection.

Use exact first-order CHUNKED vector-Jacobian products: score graph over small descriptor leaves,
then recompute DMD/UNet in chunks of2. This tests the real full loss path without retaining24 patch
activation graphs. No .backward(), optimizer, state update, higher-order gradient or training occurs.
Report raw-score, actual paired tail-risk and L1 parameter gradients, active/inactive risk and the
fixed lambda=.1 gradient ratio. Zero risk means the constraint is inactive, NOT a bug or an invented
positive risk; at least one of the six cases must have risk above its live identity-replay noise plus
5e-6 before pilot eligibility (sigmoid Lipschitz bound for1e-6 score noise at temperature.05).

Use current GPU if available (T4 expected sufficient with chunks2), otherwise CPU can be slow.
Sampling is CPU-only before transfer; GPU backward never includes grid_sample. Actual peak CUDA
memory/time is reported, not promised. This does not certify training memory or runtime.
After reset, Run All00_prepare_after_reset on this same runtime, then run35.
Completed23/33/34 artifacts are reused from Drive; do NOT rerun those notebooks. NBIS is not needed.

User approved identical aligned crops for all four arms, original23 common warmstart and2epochs/arm.
Using all anchors increases patch dose/compute relative to old single-patch epochs; any later pilot
must match this dose across ALL arms and state that it is not compute-equivalent to old23/30.
This notebook starts no pilot. If this ONE alternative fails its checks, STOP this loss direction.
Even eligibility is only permission to prepare a new controlled pilot, not novelty/Q3 evidence.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import shutil
import sys
import time
import types
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from google.colab import drive
DEVICE = torch.device('cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M1_DIR = PROJECT_ROOT / 'experiments' / 'm1_shared_curriculum_v1_seed20261110' / 'dd1427cfad6621c577bf'
SEED = 20261110
RUN_SEEDS = {'dry_l1_seed20261110': 20261110}
RUN_NAMES = tuple(RUN_SEEDS)
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
SPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'
LAMBDA_GRADIENT = 0.1
IDENTITY_FRACTION = 0.1

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact changed: {directory / name}')
print('Read-only prerequisites on CPU; DMD inference device selected below.')
MATCHER_ROOT = PROJECT_ROOT / 'experiments' / 'risk_patch_matcher_v1'
MATCHER_SETTINGS = {'epochs': 6, 'steps_per_epoch': 200, 'identities_per_batch': 12,
    'learning_rate': 0.0002, 'embedding_dim': 128, 'held_groups_per_db': 8,
    'quality_gate': {'clean_auc': 0.80, 'dry_auc': 0.70, 'clean_tar': 0.25, 'dry_tar': 0.10}}

import ast
import io
import sys
import subprocess
import urllib.request
import zipfile
from scipy.optimize import linear_sum_assignment
import scipy
import torch.nn.functional as F

INFERENCE_DEVICE = torch.device('cpu')
SOURCE33_DIR = PROJECT_ROOT / 'diagnostics' / 'pretrained_dmd_feasibility_v1' / '411576c19ecb440b8392'
print('34: cached local scalar and one live input-gradient check on CPU; no training/GPU/NBIS.')

SOURCE34_DIR = PROJECT_ROOT/'diagnostics'/'dmd_local_surrogate_v1'/'af30fd0800005636bbb8'
CHECK_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHUNK_SIZE = 2
print('35 frozen matching; live check device:',CHECK_DEVICE,'| chunk size:',CHUNK_SIZE,'| NO TRAINING')


In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_config = json.loads((M1_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_completion = json.loads((M1_DIR / 'completion.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_sha, m1_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M1_DIR))
verify_files(M0_DIR, m0_config['source_file_sha256'])
verify_files(M1_DIR, m1_config['source_file_sha256'])
verify_files(M1_DIR, m1_completion['file_sha256'])
if (m1_completion['status'] != 'shared_curriculum_control_complete' or m1_completion['configuration_sha256'] != m1_sha
        or m1_config['m0_configuration_sha256'] != m0_sha):
    raise RuntimeError('Frozen completed M1 provenance mismatch.')
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (m0_config, sim_config, pair_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or m1_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Frozen simulator/normalization provenance mismatch.')
for config in (m0_config, m1_config):
    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']
            or np.__version__ != config['numpy_version'] or Image.__version__ != config['pillow_version']):
        raise RuntimeError('Library versions differ from frozen training.')
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):
    if file_digest(M1_DIR / name) != file_digest(M0_DIR / name):
        raise RuntimeError(f'M1 model/source differs from M0: {name}')
SETTINGS = dict(m1_config['settings'])
EPOCHS = len(m1_config['epoch_conditions'])
if (EPOCHS != sum(SETTINGS['phase_lengths']) or EPOCHS != m0_config['epochs']
        or SETTINGS['base_channels'] != m0_config['base_channels'] or SETTINGS['patch_size'] != m0_config['patch_size']
        or SETTINGS['batch_size'] != m0_config['batch_size'] or SETTINGS['learning_rate'] != m0_config['learning_rate']):
    raise RuntimeError('Budget/architecture/optimizer differ from the frozen baseline.')
train_sources = pd.read_csv(M0_DIR / 'train_sources.csv', keep_default_na=False)
validation_sources = pd.read_csv(M0_DIR / 'validation_sources.csv', keep_default_na=False)
for split_name, frame in (('train', train_sources), ('validation', validation_sources)):
    allowed = locked[locked['split'].eq(split_name)].set_index('relative_path')
    if frame.empty or frame['relative_path'].duplicated().any() or frame.groupby(['dataset', 'db']).ngroups != 6 or set(frame['dataset']) != {'FVC2000', 'FVC2002'}:
        raise RuntimeError(f'Invalid source-only {split_name} data.')
    for record in frame.to_dict('records'):
        relative = record['relative_path']
        if (relative not in allowed.index or allowed.loc[relative, 'sha256'] != record['sha256']
                or allowed.loc[relative, 'identity_key'] != record['identity_key']):
            raise RuntimeError(f'Split/source mismatch: {relative}')
        path = DATA_ROOT / relative
        if split_name == 'train' and (not path.is_file() or file_digest(path) != record['sha256']):
            raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02.')
if (set(train_sources['identity_key']) & set(validation_sources['identity_key'])
        or set(train_sources['sha256']) & set(validation_sources['sha256'])
        or not validation_sources.groupby(['dataset', 'db']).size().eq(12).all()):
    raise RuntimeError('Train/validation overlap or changed validation coverage.')
training = load_module('m2_original_model_v1', M0_DIR / 'restoration_baseline.py')
simulator = load_module('m2_original_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
curriculum_data = load_module('m2_frozen_task_data_v1', M1_DIR / 'curriculum_data.py')
FROZEN_ENGINE_SOURCE = (M1_DIR / 'training_helpers.py.txt').read_bytes().decode('utf-8')
SOURCE_COUNT = len(train_sources)
all_conditions = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
print('Verified unchanged sources:', SOURCE_COUNT, 'train /', len(validation_sources), 'validation.')

allowed_train_paths = frozenset(train_sources.relative_path)


In [ ]:

pointer = json.loads((MATCHER_ROOT / 'completed_experiment.json').read_text(encoding='utf-8'))
MATCHER_DIR = Path(pointer['directory'])
if (MATCHER_DIR.parent != MATCHER_ROOT
        or file_digest(MATCHER_DIR / 'configuration.json') != pointer['configuration_sha256']
        or file_digest(MATCHER_DIR / 'completion.json') != pointer['completion_sha256']):
    raise RuntimeError('Notebook29 published pointer/integrity mismatch.')
configuration = json.loads((MATCHER_DIR / 'configuration.json').read_text(encoding='utf-8'))
completion = json.loads((MATCHER_DIR / 'completion.json').read_text(encoding='utf-8'))
if (configuration['version'] != 'risk_patch_matcher_v1'
        or configuration['settings'] != MATCHER_SETTINGS or configuration['seed'] != SEED
        or completion['status'] != 'risk_matcher_fit_complete'
        or completion['configuration_sha256'] != pointer['configuration_sha256']
        or completion['completed_epochs'] != MATCHER_SETTINGS['epochs']
        or configuration['selection'] != 'last fixed-budget checkpoint; no gate/validation epoch selection'
        or configuration['train_sources_sha256'] != file_digest(M0_DIR / 'train_sources.csv')
        or configuration['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json')
        or configuration['simulator_sha256'] != file_digest(SIM_DIR / 'fingerprint_degradation.py')
        or configuration['patch_size'] != SETTINGS['patch_size']):
    raise RuntimeError('Expected unchanged completed29 fixed-budget matcher configuration.')
verify_files(MATCHER_DIR, completion['file_sha256'])
for key, actual in (('torch_version', str(torch.__version__)), ('torch_cuda', str(torch.version.cuda)),
        ('numpy_version', np.__version__), ('pillow_version', Image.__version__)):
    if configuration[key] != actual:
        raise RuntimeError(f'Matcher library version differs: {key}')
matcher_source = (MATCHER_DIR / 'matcher_source.py').read_bytes().decode('utf-8')
if (hashlib.sha256(matcher_source.encode()).hexdigest() != configuration['model_source_sha256']
        or file_digest(MATCHER_DIR / 'inner_partition.csv') != configuration['partition_sha256']
        or file_digest(MATCHER_DIR / 'checkpoint_helpers.py.txt') != configuration['checkpoint_source_sha256']):
    raise RuntimeError('Frozen matcher source/partition hash differs.')
matcher = types.ModuleType('read_only_failed29_matcher')
matcher.__dict__.update(globals())
exec(compile(matcher_source, str(MATCHER_DIR / 'matcher_source.py'), 'exec'), matcher.__dict__)
partition = pd.read_csv(MATCHER_DIR / 'inner_partition.csv', keep_default_na=False)
expected_partition = matcher.inner_partition(train_sources, MATCHER_SETTINGS['held_groups_per_db'])
pd.testing.assert_frame_equal(partition, expected_partition, check_dtype=False)
latest = json.loads((MATCHER_DIR / 'latest.json').read_text(encoding='utf-8'))
if (latest != completion['checkpoint'] or Path(latest['file']).name != latest['file']
        or latest['configuration_sha256'] != pointer['configuration_sha256']
        or latest['completed_epochs'] != MATCHER_SETTINGS['epochs']
        or file_digest(MATCHER_DIR / latest['file']) != latest['sha256']):
    raise RuntimeError('LAST completed29 checkpoint differs.')
state = torch.load(MATCHER_DIR / latest['file'], map_location='cpu', weights_only=True)
if (state['configuration_sha256'] != pointer['configuration_sha256']
        or state['completed_epochs'] != MATCHER_SETTINGS['epochs']):
    raise RuntimeError('Saved matcher payload differs.')
with torch.random.fork_rng(devices=[]):
    model = matcher.PatchMatcher().to(DEVICE)
model.load_state_dict(state['model'], strict=True)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
if any(not torch.isfinite(value).all() for value in model.state_dict().values()):
    raise RuntimeError('Nonfinite frozen matcher state.')
saved_state = {key: value.clone() for key, value in model.state_dict().items()}
saved_history = pd.DataFrame(state['history'])
if saved_history.epoch.tolist() != list(range(1, MATCHER_SETTINGS['epochs'] + 1)):
    raise RuntimeError('Matcher has not trained the full predefined budget.')
del state
calibration = pd.read_csv(MATCHER_DIR / 'calibration_scores.csv', float_precision='round_trip')
gate = pd.read_csv(MATCHER_DIR / 'gate_scores.csv', float_precision='round_trip')
saved_gate = json.loads((MATCHER_DIR / 'quality_gate.json').read_text(encoding='utf-8'))
saved_thresholds = json.loads((MATCHER_DIR / 'thresholds.json').read_text(encoding='utf-8'))
if saved_gate != completion['quality_gate']:
    raise RuntimeError('Saved29 gate attestation differs.')
print('Read-only loaded matcher epoch:', latest['completed_epochs'], '| original gate passed:', saved_gate['passed'])


In [ ]:
AUDIT_SOURCE = 'def panel_records(partition, split):\n    if split not in (\'fit\', \'gate\', \'calibration\'):\n        raise ValueError(\'Only explicit main-TRAIN inner partitions are supported.\')\n    selected = partition[partition.matcher_split.eq(split)]\n    if selected.empty:\n        raise RuntimeError(\'Missing inner partition.\')\n    if split == \'fit\':\n        frames = []\n        for (dataset, db), rows in selected.groupby([\'dataset\', \'db\'], sort=True):\n            groups = sorted(rows.connected_group.unique(),\n                key=lambda key: training.stable_seed(f\'{SEED}|audit_fit_panel|{key}\'))\n            if len(groups) < MATCHER_SETTINGS[\'held_groups_per_db\']:\n                raise RuntimeError(\'Insufficient fit groups for matched audit panel.\')\n            frames.append(rows[rows.connected_group.isin(groups[:MATCHER_SETTINGS[\'held_groups_per_db\']])])\n        selected = pd.concat(frames, ignore_index=True)\n    galleries, probes = [], []\n    for key, rows in selected.groupby(\'identity_key\', sort=True):\n        records = rows.sort_values(\'relative_path\').drop_duplicates(\'sha256\').to_dict(\'records\')\n        if len(records) < 3:\n            raise RuntimeError(f\'Audit requires the same three byte-distinct source impressions: {key}\')\n        galleries.append(records[0])\n        probes.extend(records[1:3])\n    return galleries, probes\n\ndef pair_metadata(partition, split):\n    galleries, probes = panel_records(partition, split)\n    rows = []\n    for condition in (\'identity/none\', \'dry_like/mild\', \'dry_like/moderate\', \'dry_like/severe\'):\n        for probe in probes:\n            for gallery in galleries:\n                if (probe[\'dataset\'], probe[\'db\']) != (gallery[\'dataset\'], gallery[\'db\']):\n                    continue\n                label = int(probe[\'identity_key\'] == gallery[\'identity_key\'])\n                if not label and probe[\'connected_group\'] == gallery[\'connected_group\']:\n                    continue\n                rows.append(dict(dataset=probe[\'dataset\'], db=probe[\'db\'], condition=condition,\n                    probe=probe[\'relative_path\'], gallery=gallery[\'relative_path\'], label=label))\n    return pd.DataFrame(rows)\n\ndef verify_score_metadata(frame, partition, split):\n    keys = [\'dataset\', \'db\', \'condition\', \'probe\', \'gallery\', \'label\']\n    if (frame.empty or frame.duplicated(keys).any() or not np.isfinite(frame.score).all()\n            or not frame.score.between(-1.00001, 1.00001).all() or set(frame.label) != {0, 1}):\n        raise RuntimeError(\'Invalid saved inner verification scores.\')\n    expected = pair_metadata(partition, split)\n    pd.testing.assert_frame_equal(frame[keys].sort_values(keys).reset_index(drop=True),\n        expected[keys].sort_values(keys).reset_index(drop=True), check_dtype=False)\n\ndef crop_details(record):\n    relative = record[\'relative_path\']\n    if relative not in allowed_train_paths:\n        raise RuntimeError(\'Crop audit can read main TRAIN images only.\')\n    dpi = pair_config[\'source_dpi\'][f"{record[\'dataset\']}_{record[\'db\']}"]\n    reference = training.normalized_image(str(DATA_ROOT / relative), dpi)\n    foreground = simulator.foreground_mask(reference)\n    ys, xs = np.where(foreground)\n    if not len(ys):\n        raise RuntimeError(f\'No foreground for frozen crop: {relative}\')\n    crop_seed = training.stable_seed(f\'{SEED}|0|{relative}|crop\')\n    rng = np.random.default_rng(crop_seed)\n    anchor = int(rng.integers(len(ys)))\n    size = SETTINGS[\'patch_size\']\n    height, width = reference.shape\n    top = int(np.clip(int(ys[anchor]) - size // 2, 0, max(0, height - size)))\n    left = int(np.clip(int(xs[anchor]) - size // 2, 0, max(0, width - size)))\n    pad = ((0, max(0, size - height)), (0, max(0, size - width)))\n    padded = np.pad(reference, pad, mode=\'reflect\') if any(value for pair in pad for value in pair) else reference\n    patch = np.array(padded[top:top + size, left:left + size], dtype=np.float32, copy=True)\n    actual = matcher.sample_patches([record], 0, [(\'identity\', \'none\')])[0, 0].numpy()\n    if not np.array_equal(patch, actual):\n        raise RuntimeError(\'Reconstructed crop is not byte-identical to frozen29 sampling.\')\n    coverage = (min(size, height - top) * min(size, width - left)) / (height * width)\n    metadata = dict(relative_path=relative, identity_key=record[\'identity_key\'], dataset=record[\'dataset\'],\n        db=record[\'db\'], crop_seed=crop_seed, normalized_height=height, normalized_width=width,\n        top=top, left=left, patch_size=size, padded_bottom=pad[0][1], padded_right=pad[1][1],\n        crop_image_fraction=coverage, crop_foreground_fraction=float(\n            foreground[top:min(top + size, height), left:min(left + size, width)].mean()),\n        crop_sha256=hashlib.sha256(patch.astype(\'<f4\').tobytes()).hexdigest())\n    return metadata, reference, patch\n\ndef describe_scores(values):\n    values = np.asarray(values, dtype=np.float64)\n    if not len(values) or not np.isfinite(values).all():\n        raise RuntimeError(\'Nonempty finite scores required for descriptive summaries.\')\n    return dict(count=len(values), mean=float(values.mean()), std=float(values.std(ddof=1)) if len(values) > 1 else None,\n        q05=float(np.quantile(values, .05)), median=float(np.median(values)), q95=float(np.quantile(values, .95)))\n\ndef auc_against(genuine, impostors):\n    genuine, impostors = np.asarray(genuine), np.asarray(impostors)\n    if not len(genuine) or not len(impostors):\n        raise RuntimeError(\'Both positive and negative scores required.\')\n    return float(np.mean((genuine[:, None] > impostors).astype(float)\n        + .5 * (genuine[:, None] == impostors)))\n\n@torch.inference_mode()\ndef embed_panel(records, condition):\n    features = []\n    for start in range(0, len(records), 12):\n        images = matcher.sample_patches(records[start:start + 12], 0, [condition]).to(DEVICE)\n        value = model(images)\n        if not torch.isfinite(value).all():\n            raise RuntimeError(\'Nonfinite CPU embeddings.\')\n        features.append(value.cpu())\n    return torch.cat(features)\n\ndef summarize_panel(frame):\n    summaries = []\n    for (split, dataset, db, condition), group in frame.groupby(\n            [\'inner_split\', \'dataset\', \'db\', \'condition\'], sort=True):\n        threshold = saved_thresholds[f\'{dataset}_{db}\']\n        positive = group[group.pool.eq(\'different_impression_genuine\')].score.to_numpy()\n        for pool in (\'different_impression_genuine\', \'same_source_control\', \'same_db_impostor\', \'cross_db_impostor\'):\n            values = group[group.pool.eq(pool)].score.to_numpy()\n            summaries.append(dict(inner_split=split, dataset=dataset, db=db, condition=condition, pool=pool,\n                threshold=threshold, acceptance_fraction=float(np.mean(values >= threshold)),\n                auc_against_cross_impression_genuine=auc_against(positive, values)\n                    if pool.endswith(\'impostor\') else None, **describe_scores(values)))\n    return pd.DataFrame(summaries)\n'
exec(compile(AUDIT_SOURCE, '<matcher-failure-audit>', 'exec'))


In [ ]:

verify_score_metadata(calibration, partition, 'calibration')
verify_score_metadata(gate, partition, 'gate')
thresholds, replay_metrics, replay_summary = matcher.gate_analysis(calibration, gate)
if saved_thresholds != {f'{dataset}_{db}': value for (dataset, db), value in thresholds.items()}:
    raise RuntimeError('Exact CSV replay changed original calibration thresholds.')
if replay_summary != saved_gate:
    raise RuntimeError('Saved gate summary differs from exact saved-score replay.')
prior_metrics = pd.read_csv(MATCHER_DIR / 'gate_metrics.csv', float_precision='round_trip')
pd.testing.assert_frame_equal(replay_metrics, prior_metrics, check_dtype=False, check_exact=True)
print('Exact saved-score replay verified; gate UNCHANGED:')
print(json.dumps(replay_summary, indent=2))
print('Saved fixed-budget learning/runtime/memory history:')
print(saved_history.to_string(index=False))


In [ ]:

EXPECTED_DMD_FILES = {'models/model_zoo.py': '39f24735c03bcbbc0d51f6af75146bd3beb55a17bc25cf1a3b1c8f99bdd54b29', 'models/resnet.py': 'e6d28ed8a9730fc0ce5f1a98d464a129cd976ad91028e4516880908b42688e8d', 'models/units.py': '3c685be67e6ad57c5016cf44f75aa7660ba4675f17277faec470d44b3215a1aa', 'models/inception.py': 'bac60ed9f06c9e2e0869f73bb2bf987bbd3889856e4509f9a0cec5070d05702d', 'models/dataloader_densemnt.py': '70308d0b9a6258f5fb2eb815fa296a987190c0db8d9e5750c00c9df16e9468aa', 'evaluate_mnt.py': '84e7f4e62ebdaf718bc1266367cee5fa7680fd599d585c3cc9398f35a4e5b78f', 'README.md': '2b538fe64ef3510f60512a7b413eb04bbfc66a4c51b4fa00aa67e533eb84ee68', 'LICENCE': '4878c4e24ae4dd9878852724b4ae3313dc9a0599680d6552c1606bbc3d63830c', 'DMD.yaml': '12186bc0a33ab479d9c041aae8e35a180a286b98500c3d990475403732538b3d'}
EXPECTED_DMD_COMMIT = 'feb0ad40e5f340caebb457237d7ef21d5d5724e6'
EXPECTED_WEIGHT_SHA = '6d89dff9e5502cf4534b0fd9e014b3fb4a494a0be904015319c72edd9a7aff48'
EXPECTED_ADAPTER_SHA = '82cde5da161f7c37b58fe1ce069b460965c2a94eb175d12405840fffdc6c0919'
source_config = json.loads((SOURCE33_DIR / 'configuration.json').read_text(encoding='utf-8'))
source33_sha = file_digest(SOURCE33_DIR / 'configuration.json')
source_completion = json.loads((SOURCE33_DIR / 'completion.json').read_text(encoding='utf-8'))
if (source_completion['status'] != 'pretrained_dmd_feasibility_complete'
        or source_completion['configuration_sha256'] != source33_sha
        or SOURCE33_DIR.name != source33_sha[:20]
        or source_config['version'] != 'pretrained_dmd_feasibility_v1'
        or source_config['source_commit'] != EXPECTED_DMD_COMMIT
        or source_config['source_files'] != EXPECTED_DMD_FILES
        or source_config['weight_sha256'] != EXPECTED_WEIGHT_SHA
        or source_config['weight_bytes'] != 179093287
        or source_config['max_anchors'] != 24 or source_config['minimum_anchors'] != 4
        or source_config['adapter_sha256'] != EXPECTED_ADAPTER_SHA
        or source_config['matcher29_configuration_sha256'] != pointer['configuration_sha256']
        or source_config['matcher29_completion_sha256'] != pointer['completion_sha256']
        or source_completion['permits_restoration_training'] is not False):
    raise RuntimeError('Expected the unchanged completed33 source, not a different run/checkpoint.')
verify_files(SOURCE33_DIR,source_completion['file_sha256'])
for name,value in [('torch_version',str(torch.__version__)),('torch_cuda',str(torch.version.cuda)),
        ('numpy_version',np.__version__),('pillow_version',Image.__version__),('scipy_version',scipy.__version__)]:
    if source_config[name] != value:
        raise RuntimeError(f'Frozen33 library differs: {name}')
ASSET_ROOT = PROJECT_ROOT / 'pretrained' / 'dmd_sd14_v1' / EXPECTED_DMD_COMMIT
verify_files(ASSET_ROOT,EXPECTED_DMD_FILES)
WEIGHT_PATH = ASSET_ROOT / 'best_model.pth.tar'
if file_digest(WEIGHT_PATH) != EXPECTED_WEIGHT_SHA or WEIGHT_PATH.stat().st_size != 179093287:
    raise RuntimeError('Verified public DMD weights missing/changed; no unsafe fallback.')
adapter33_source = (SOURCE33_DIR / 'adapter.py.txt').read_bytes().decode('utf-8')
if hashlib.sha256(adapter33_source.encode()).hexdigest() != EXPECTED_ADAPTER_SHA:
    raise RuntimeError('Frozen33 adapter changed.')
exec(compile(adapter33_source,'<unchanged33-adapter>','exec'))
local_score,_unused_lsar = load_scoring(ASSET_ROOT / 'evaluate_mnt.py')
del _unused_lsar
dmd_model = load_dmd(ASSET_ROOT,WEIGHT_PATH)
initial_dmd_sha = state_sha(dmd_model)
print('Verified and loaded frozen33:',SOURCE33_DIR.name,'| previous LSA-R gate:',source_completion['quality']['passed'])


In [ ]:
LOCAL_SOURCE = "def query_anchor_index(relative, count, epoch=0):\n    if count < 4:\n        raise RuntimeError('Local surrogate needs at least4 original anchors, same as33.')\n    return training.stable_seed(f'{SEED}|{epoch}|{relative}|dmd_anchor_v1') % count\n\ndef read_cached33(record, family, severity):\n    relative = record['relative_path']\n    if relative not in allowed_train_paths:\n        raise RuntimeError('Cached DMD local analysis is main TRAIN only.')\n    directory = SOURCE33_DIR / 'anchors' / hashlib.sha256(relative.encode()).hexdigest()[:24]\n    expected = dict(relative_path=relative,source_sha256=record['sha256'],\n        configuration_sha256=source33_sha,mindtct_sha256=source_config['nbis_build']['binary_sha256']['mindtct'])\n    if cached_bundle(directory,expected) is None:\n        raise RuntimeError(f'Missing completed33 anchor cache: {relative}. Run33; do not silently re-extract.')\n    with np.load(directory / 'anchors.npz',allow_pickle=False) as archive:\n        anchors = archive['anchors'].copy()\n        shape = archive['image_shape'].copy()\n    parsed = parse_nbis_xyt((directory / 'minutiae.xyt').read_text(encoding='ascii'),*shape)\n    if not np.array_equal(anchors,parsed):\n        raise RuntimeError('Frozen anchors disagree with their XYT source.')\n    if len(anchors) < 4:\n        return dict(status='unmatchable_original_anchor_count',anchors=anchors,feature=None,mask=None)\n    feature_key = hashlib.sha256(f'{relative}|{family}|{severity}'.encode()).hexdigest()[:24]\n    directory = SOURCE33_DIR / 'features' / feature_key\n    expected = dict(relative_path=relative,family=family,severity=severity,\n        configuration_sha256=source33_sha,anchors_sha256=hashlib.sha256(anchors.tobytes()).hexdigest())\n    if cached_bundle(directory,expected) is None:\n        raise RuntimeError(f'Missing completed33 feature cache: {relative}/{family}/{severity}. Run33.')\n    with np.load(directory / 'features.npz',allow_pickle=False) as archive:\n        feature = torch.from_numpy(archive['feature'].copy())\n        mask = torch.from_numpy(archive['mask'].copy())\n    if (feature.shape != (len(anchors),768) or mask.shape != (len(anchors),64)\n            or not torch.isfinite(feature).all() or not torch.isfinite(mask).all()\n            or not ((mask>=0)&(mask<=1)).all()):\n        raise RuntimeError('Invalid cached33 local features.')\n    return dict(status='matched',anchors=anchors,feature=feature,mask=mask)\n\ndef local_identity_score(query_feature,query_mask,gallery_feature,gallery_mask):\n    if (query_feature.shape!=(1,768) or query_mask.shape!=(1,64)\n            or gallery_feature.ndim!=2 or gallery_feature.shape[1]!=768\n            or gallery_mask.shape!=(len(gallery_feature),64) or len(gallery_feature)<4\n            or len(gallery_feature)>24):\n        raise RuntimeError('Expected one local query and one4..24-anchor identity template.')\n    for value in (query_feature,query_mask,gallery_feature,gallery_mask):\n        if not torch.isfinite(value).all():\n            raise RuntimeError('Nonfinite local surrogate input.')\n    matrix = local_score(query_feature[None],gallery_feature[None],query_mask[None],gallery_mask[None],\n        ndim_feat=12,Normalize=False,binary=False)[0]\n    result = matrix.max(dim=1).values.squeeze(0)\n    if not torch.isfinite(result):\n        raise RuntimeError('Nonfinite local identity score.')\n    return result\n\ndef aligned_training_patch(image,anchor):\n    if image.ndim!=4 or image.shape[:2]!=(1,1) or not torch.isfinite(image).all():\n        raise RuntimeError('Expected one finite N1HW source.')\n    point = torch.as_tensor(anchor,device=image.device,dtype=image.dtype).flatten()\n    if len(point)<3 or not torch.isfinite(point).all():\n        raise RuntimeError('Expected a finite fixed original anchor.')\n    height,width = image.shape[-2:]\n    yy,xx = torch.meshgrid(torch.arange(192,device=image.device,dtype=image.dtype)-96,\n        torch.arange(192,device=image.device,dtype=image.dtype)-96,indexing='ij')\n    theta = torch.deg2rad(point[2])\n    sx = point[0]+torch.cos(theta)*xx-torch.sin(theta)*yy\n    sy = point[1]+torch.sin(theta)*xx+torch.cos(theta)*yy\n    grid = torch.stack(((2*sx+1)/width-1,(2*sy+1)/height-1),dim=-1)[None]\n    return F.grid_sample(image-.5,grid,mode='bilinear',padding_mode='zeros',align_corners=False)+.5\n\ndef local_prediction_features(prediction):\n    if (prediction.ndim!=4 or prediction.shape[1:]!=(1,192,192)\n            or not torch.isfinite(prediction).all()):\n        raise RuntimeError('Expected finite N1x192x192 restoration outputs.')\n    output = dmd_model.get_embedding(2*prediction[:,:,32:160,32:160]-1)\n    return output['feature'],output['mask']\n\ndef counterfactual_risks(before,after,threshold,temperature=.05,tail_fraction=.1):\n    if (before.ndim!=1 or before.shape!=after.shape or not len(before)\n            or not torch.isfinite(before).all() or not torch.isfinite(after).all()\n            or not math.isfinite(threshold) or temperature<=0 or not 0<tail_fraction<=1):\n        raise RuntimeError('Finite matched nonmated score vectors and valid risk constants required.')\n    increase = torch.relu(torch.sigmoid((after-threshold)/temperature)\n        -torch.sigmoid((before.detach()-threshold)/temperature))\n    count = max(1,math.ceil(tail_fraction*len(increase)))\n    return increase.mean(),increase.topk(count).values.mean()\n\ndef selected_query(record,bundle):\n    if bundle['status']!='matched':\n        raise RuntimeError('Unmatchable queries must be explicitly counted upstream.')\n    index = query_anchor_index(record['relative_path'],len(bundle['anchors']),0)\n    return index,bundle['feature'][index:index+1],bundle['mask'][index:index+1]\n"
exec(compile(LOCAL_SOURCE,'<dmd-local-scalar>','exec'))


In [ ]:

source34_config=json.loads((SOURCE34_DIR/'configuration.json').read_text(encoding='utf-8'))
source34_completion=json.loads((SOURCE34_DIR/'completion.json').read_text(encoding='utf-8'))
source34_sha=file_digest(SOURCE34_DIR/'configuration.json')
if (source34_completion['status']!='dmd_local_surrogate_complete'
        or SOURCE34_DIR.name!=source34_sha[:20]
        or source34_completion['configuration_sha256']!=source34_sha
        or source34_config['source33_configuration_sha256']!=source33_sha
        or source34_completion['permits_legacy_30_31'] is not False):
    raise RuntimeError('Expected unchanged completed34; no gate rewriting permitted.')
verify_files(SOURCE34_DIR,source34_completion['file_sha256'])
source34_completion_sha=file_digest(SOURCE34_DIR/'completion.json')
print('34 remains unchanged. Quality:',source34_completion['quality']['passed'])
SPECIALIZATION_ROOT=PROJECT_ROOT/'experiments'/'m5_dry_specialization_v1'
original_pointer=json.loads((SPECIALIZATION_ROOT/'completed_experiment.json').read_text(encoding='utf-8'))
ORIGINAL_DIR=Path(original_pointer['directory'])
if (ORIGINAL_DIR.parent!=SPECIALIZATION_ROOT
        or file_digest(ORIGINAL_DIR/'configuration.json')!=original_pointer['configuration_sha256']
        or file_digest(ORIGINAL_DIR/'completion.json')!=original_pointer['completion_sha256']):
    raise RuntimeError('Original23 pointer/configuration/completion changed.')
original_config=json.loads((ORIGINAL_DIR/'configuration.json').read_text(encoding='utf-8'))
original_completion=json.loads((ORIGINAL_DIR/'completion.json').read_text(encoding='utf-8'))
if (original_config['version']!='m5_dry_specialization_v1'
        or original_completion['status']!='dry_specialization_complete'
        or original_completion['configuration_sha256']!=original_pointer['configuration_sha256']
        or original_config['seed']!=SEED
        or original_config['initialization_seeds']!={'dry_l1_seed20261110':20261110}
        or original_config['selection_metric']!='same fixed validation mean global L1; initial identity checkpoint included'
        or original_config['losses']!={'dry_l1_seed20261110':'global L1'}
        or original_config['gradient_lambda']!=0
        or original_config['settings']!=SETTINGS):
    raise RuntimeError('Expected the original23 common seed10 L1 warmstart, without reselection.')
verify_files(ORIGINAL_DIR,original_completion['file_sha256'])
verify_files(ORIGINAL_DIR,original_config['source_file_sha256'])
for name,path in [('restoration_baseline.py',M0_DIR/'restoration_baseline.py'),
        ('train_sources.csv',M0_DIR/'train_sources.csv')]:
    if original_config['source_file_sha256'][name]!=file_digest(path):
        raise RuntimeError(f'Original23 source changed: {name}')
if original_config['simulator_configuration_sha256']!=file_digest(SIM_DIR/'configuration.json'):
    raise RuntimeError('Original23 simulator differs.')
for name,value in [('torch_version',str(torch.__version__)),('torch_cuda',str(torch.version.cuda)),
        ('numpy_version',np.__version__),('pillow_version',Image.__version__)]:
    if original_config[name]!=value:
        raise RuntimeError(f'Original23 runtime differs: {name}')
original_run=ORIGINAL_DIR/'dry_l1_seed20261110'
if file_digest(original_run/'completion.json')!=original_completion['run_completion_sha256']['dry_l1_seed20261110']:
    raise RuntimeError('Original23 selected run changed.')
original_run_completion=json.loads((original_run/'completion.json').read_text(encoding='utf-8'))
verify_files(original_run,original_run_completion['file_sha256'])
original_best=json.loads((original_run/'best.json').read_text(encoding='utf-8'))
if (original_best!=original_run_completion['result']['best_checkpoint']
        or Path(original_best['file']).name!=original_best['file']
        or original_best['configuration_sha256']!=original_pointer['configuration_sha256']
        or file_digest(original_run/original_best['file'])!=original_best['sha256']
        or original_run_completion['epochs_completed']!=8):
    raise RuntimeError('Original23 best checkpoint/budget differs.')
payload=torch.load(original_run/original_best['file'],map_location='cpu',weights_only=True)
if (payload['configuration_sha256']!=original_pointer['configuration_sha256']
        or payload['run']!='dry_l1_seed20261110' or payload['initialization_seed']!=20261110
        or not 0<=payload['completed_epochs']<=8):
    raise RuntimeError('Original23 safe payload differs.')
with torch.random.fork_rng(devices=[]):
    restorer=training.RestorationUNet()
restorer.load_state_dict(payload['model'],strict=True)
del payload
restorer.eval()
if sum(parameter.numel() for parameter in restorer.parameters())!=487009:
    raise RuntimeError('Original23 restoration architecture differs.')
if any(not torch.isfinite(value).all() for value in restorer.state_dict().values()):
    raise RuntimeError('Nonfinite original23 weights.')
initial_restorer_sha=state_sha(restorer)
print('Verified original23 selected warmstart. No optimizer or updates.')


In [ ]:
FROZEN_SOURCE = "def freeze_lsar(matrix,query_anchors,gallery_anchors):\n    if (matrix.ndim!=2 or matrix.shape!=(len(query_anchors),len(gallery_anchors))\n            or min(matrix.shape)<4 or max(matrix.shape)>24 or not torch.isfinite(matrix).all()):\n        raise RuntimeError('Finite4..24-anchor local matrices required for frozen LSA-R.')\n    leaf=matrix.detach().clone().requires_grad_()\n    value=lsar_score(leaf[None],query_anchors[None,:,:3],gallery_anchors[None,:,:3])[0]\n    weights,=torch.autograd.grad(value,leaf)\n    weights=weights.detach()\n    baseline=value.detach()\n    residual=float((baseline-(weights*leaf.detach()).sum()).abs())\n    if (not torch.isfinite(weights).all() or not torch.isfinite(baseline)\n            or bool((weights<0).any()) or float(weights.sum())<=0):\n        raise RuntimeError('Invalid fixed LSA-R derivative/weights.')\n    return dict(before=leaf.detach(),weights=weights,baseline=baseline,homogeneity_residual=residual)\n\ndef frozen_lsar_score(plan,after_matrix):\n    if after_matrix.shape!=plan['before'].shape or not torch.isfinite(after_matrix).all():\n        raise RuntimeError('Fixed before/after local matrix shape/values differ.')\n    device=after_matrix.device\n    return plan['baseline'].to(device)+(plan['weights'].to(device)*\n        (after_matrix-plan['before'].to(device))).sum()\n\ndef local_matrix(feature,mask,target):\n    device=feature.device\n    return local_score(feature[None],target['feature'].to(device)[None],\n        mask[None],target['mask'].to(device)[None],ndim_feat=12,Normalize=False,binary=False)[0]\n\ndef named_train_panels():\n    panels={}\n    for split in ('calibration','gate'):\n        galleries,probes=panel_records(partition,split)\n        panels[split]=(galleries,probes)\n    return panels\n\ndef all_aligned_patches(array,anchors):\n    image=torch.from_numpy(array.copy())[None,None]\n    return torch.cat([aligned_training_patch(image,anchor) for anchor in anchors])\n\n@torch.no_grad()\ndef patch_descriptors(patches,apply_restorer):\n    features,masks=[],[]\n    for start in range(0,len(patches),CHUNK_SIZE):\n        value=patches[start:start+CHUNK_SIZE].to(CHECK_DEVICE)\n        prediction=restorer(value) if apply_restorer else value\n        value=dmd_model.get_embedding(2*prediction[:,:,32:160,32:160]-1)\n        features.append(value['feature'].detach())\n        masks.append(value['mask'].detach())\n    return torch.cat(features),torch.cat(masks)\n\ndef parameter_vjps(patches,targets,feature_grads,mask_grads):\n    parameters=tuple(restorer.parameters())\n    sums=[[torch.zeros_like(p) for p in parameters] for _ in feature_grads]\n    l1_sums=[torch.zeros_like(p) for p in parameters]\n    prediction_l1=[0.0 for _ in feature_grads]\n    finite=True\n    l1_value=0.0\n    for start in range(0,len(patches),CHUNK_SIZE):\n        end=min(start+CHUNK_SIZE,len(patches))\n        prediction=restorer(patches[start:end].to(CHECK_DEVICE))\n        values=dmd_model.get_embedding(2*prediction[:,:,32:160,32:160]-1)\n        for objective,(df,dm) in enumerate(zip(feature_grads,mask_grads)):\n            grads=torch.autograd.grad((values['feature'],values['mask']),\n                (prediction,*parameters),grad_outputs=(df[start:end],dm[start:end]),\n                retain_graph=True)\n            finite=finite and all(bool(torch.isfinite(grad).all()) for grad in grads)\n            prediction_l1[objective]+=float(grads[0].abs().sum())\n            for accumulated,grad in zip(sums[objective],grads[1:]):\n                accumulated.add_(grad.detach())\n        l1=(prediction-targets[start:end].to(CHECK_DEVICE)).abs().mean()*((end-start)/len(patches))\n        l1_value+=float(l1.detach())\n        grads=torch.autograd.grad(l1,parameters)\n        finite=finite and all(bool(torch.isfinite(grad).all()) for grad in grads)\n        for accumulated,grad in zip(l1_sums,grads):\n            accumulated.add_(grad.detach())\n    norms=[dict(parameter_l1=sum(float(g.abs().sum()) for g in grads),\n        parameter_l2=math.sqrt(sum(float(g.square().sum()) for g in grads)),\n        prediction_l1=prediction_l1[index]) for index,grads in enumerate(sums)]\n    l1_norm=math.sqrt(sum(float(g.square().sum()) for g in l1_sums))\n    return dict(finite=finite,objectives=norms,l1_parameter_l2=l1_norm,l1_value=l1_value)\n"
exec(compile(FROZEN_SOURCE,'<frozen-before-lsar>','exec'))


In [ ]:

implementation_sha='4dbeb3d839da81b64081fc9aecc070587f9bc60d010cc3287ea5ccfe52e8736d'
configuration35=dict(version='frozen_before_lsar_check_v1',implementation_sha256=implementation_sha,
    source_sha256=hashlib.sha256(FROZEN_SOURCE.encode()).hexdigest(),
    source33_configuration_sha256=source33_sha,source33_completion_sha256=file_digest(SOURCE33_DIR/'completion.json'),
    source34_configuration_sha256=source34_sha,source34_completion_sha256=source34_completion_sha,
    original23_configuration_sha256=original_pointer['configuration_sha256'],
    original23_completion_sha256=original_pointer['completion_sha256'],original23_best=original_best,
    score='official_before + sum(detached_before_Jacobian * (after_local_scores-before_local_scores))',
    query='all <=24 original anchors; fixed before assignment/rank/geometry',finite_difference_epsilon=.0001,
    live_cases='first calibration probe/DB; dry_like/moderate; all sameDB nonduplicate gallery identities',
    chunk_size=CHUNK_SIZE,check_device=str(CHECK_DEVICE),
    cuda_name=torch.cuda.get_device_name() if CHECK_DEVICE.type=='cuda' else None,
    temperature=.05,tail_fraction=.1,lambda_auxiliary=.1,input_replay_score_tolerance=1e-6,
    active_risk_above_replay_noise=5e-6,training=False,legacy30_31_authorized=False)
frozen_sha=hashlib.sha256(json_bytes(configuration35)).hexdigest()
FROZEN_ROOT=PROJECT_ROOT/'diagnostics'/'frozen_before_lsar_check_v1'
FROZEN_DIR=FROZEN_ROOT/frozen_sha[:20]
FROZEN_DIR.mkdir(parents=True,exist_ok=True)
for name,data in {'configuration.json':json_bytes(configuration35),'frozen_lsar.py.txt':FROZEN_SOURCE.encode()}.items():
    path=FROZEN_DIR/name
    if path.exists() and path.read_bytes()!=data:
        raise RuntimeError(f'Frozen35 configuration differs: {path}')
    if not path.exists():
        write_verified(path,data)
local_score,lsar_score=load_scoring(ASSET_ROOT/'evaluate_mnt.py')
source_thresholds=json.loads((SOURCE33_DIR/'thresholds.json').read_text(encoding='utf-8'))
panels35=named_train_panels()


In [ ]:

started=time.monotonic()
replay_frames,plan_records,plan_weights,plan_shapes,plan_baselines,plan_masses={},[],[],[],[],[]
replay_errors,homogeneity_errors=[],[]
original35={}
for split,(galleries,probes) in panels35.items():
    saved=pd.read_csv(SOURCE33_DIR/f'{split}_scores.csv',float_precision='round_trip')
    keys=['dataset','db','condition','probe','gallery','label']
    pd.testing.assert_frame_equal(saved[keys].sort_values(keys).reset_index(drop=True),
        pair_metadata(partition,split)[keys].sort_values(keys).reset_index(drop=True),check_dtype=False)
    records={record['relative_path']:record for record in galleries+probes}
    for record in galleries+probes:
        original35[record['relative_path']]=read_cached33(record,'identity','none')
    rows=[]
    for condition,group in saved.groupby('condition',sort=True):
        family,severity=condition.split('/')
        probes_cache={probe['relative_path']:read_cached33(probe,family,severity) for probe in probes}
        for row in group.to_dict('records'):
            query=probes_cache[row['probe']]
            target=original35[row['gallery']]
            matchable=query['status']=='matched' and target['status']=='matched'
            if bool(pd.notna(row['score']))!=matchable:
                raise RuntimeError('Cached33 pair matchability differs.')
            if not matchable:
                rows.append(dict(row))
                continue
            matrix=local_matrix(query['feature'],query['mask'],target)
            plan=freeze_lsar(matrix,torch.from_numpy(query['anchors']),torch.from_numpy(target['anchors']))
            error=abs(float(plan['baseline'])-row['score'])
            replay_errors.append(error)
            homogeneity_errors.append(plan['homogeneity_residual'])
            noop=frozen_lsar_score(plan,matrix)
            if float(noop)!=float(plan['baseline']):
                raise RuntimeError('Identity frozen-before counterfactual is not exactly zero.')
            padded=np.zeros((24,24),dtype=np.float32)
            h,w=matrix.shape
            padded[:h,:w]=plan['weights'].numpy()
            plan_records.append(dict(inner_split=split,**{key:row[key] for key in keys},
                before_score=float(plan['baseline']),geometry_mass=float(plan['weights'].sum()),
                active_cells=int((plan['weights']>0).sum()),query_anchors=h,gallery_anchors=w))
            plan_weights.append(padded)
            plan_shapes.append((h,w))
            plan_baselines.append(float(plan['baseline']))
            plan_masses.append(float(plan['weights'].sum()))
            rows.append(dict(row,score=float(plan['baseline'])))
        print(f'Frozen-before replay {split}/{condition}: {len(group)} pairs',flush=True)
    replay_frames[split]=pd.DataFrame(rows)
    write_csv(FROZEN_DIR/f'{split}_replayed_scores.csv',replay_frames[split])
thresholds35,metrics35,quality35=quality_analysis(replay_frames['calibration'],replay_frames['gate'])
prior_metrics=pd.read_csv(SOURCE33_DIR/'gate_metrics.csv',float_precision='round_trip')
metric_difference=float(np.nanmax(np.abs(metrics35['auc'].to_numpy()-prior_metrics['auc'].to_numpy())))
decision_changes=0
for split,frame in replay_frames.items():
    saved=pd.read_csv(SOURCE33_DIR/f'{split}_scores.csv',float_precision='round_trip')
    keys=['dataset','db','condition','probe','gallery','label']
    merged=frame.merge(saved,on=keys,suffixes=('_replayed','_saved'),validate='one_to_one')
    if len(merged)!=len(frame) or len(merged)!=len(saved):
        raise RuntimeError('Before replay omitted/duplicated pairs.')
    for row in merged.itertuples():
        threshold=source_thresholds[f'{row.dataset}_{row.db}']
        if threshold is not None:
            decision_changes+=int((row.score_replayed>=threshold)!=(row.score_saved>=threshold))
geometry_rows=[]
plan_frame=pd.DataFrame(plan_records)
for (split,dataset,db,condition),group in plan_frame.groupby(['inner_split','dataset','db','condition'],sort=True):
    positive=group[group.label.eq(1)].geometry_mass.to_numpy()
    negative=group[group.label.eq(0)].geometry_mass.to_numpy()
    geometry_rows.append(dict(inner_split=split,dataset=dataset,db=db,condition=condition,
        geometry_only_auc=auc_against(positive,negative) if len(positive) and len(negative) else None))
replay35=dict(valid_pairs=len(replay_errors),max_official_difference=max(replay_errors) if replay_errors else None,
    max_homogeneity_residual=max(homogeneity_errors) if homogeneity_errors else None,
    threshold_decision_changes=decision_changes,thresholds_exactly_equal=thresholds35==source_thresholds,
    max_auc_difference=metric_difference,source33_quality_passed=source_completion['quality']['passed'],
    geometry_only_note='Weight mass is score with all local similarities set to1. It can discriminate from frozen original geometry alone; not descriptor evidence.')
replay35['passed']=bool(replay_errors and replay35['max_official_difference']<=1e-6
    and replay35['max_homogeneity_residual']<=1e-6 and decision_changes==0
    and replay35['thresholds_exactly_equal'] and metric_difference<=1e-10)
write_csv(FROZEN_DIR/'plan_metadata.csv',plan_frame)
atomic_npz(FROZEN_DIR/'plans.npz',weights=np.stack(plan_weights),shapes=np.array(plan_shapes),
    baselines=np.array(plan_baselines),geometry_mass=np.array(plan_masses))
write_csv(FROZEN_DIR/'geometry_only_control.csv',pd.DataFrame(geometry_rows))
write_csv(FROZEN_DIR/'replayed_gate_metrics.csv',metrics35)
write_verified(FROZEN_DIR/'replay.json',json_bytes(replay35))
print(json.dumps(replay35,indent=2))
print('Gate geometry-only macro AUC:',pd.DataFrame(geometry_rows).query("inner_split=='gate'").geometry_only_auc.mean())


In [ ]:

fd_rows=[]
galleries,probes=panels35['calibration']
for dataset,db in sorted({(p['dataset'],p['db']) for p in probes}):
    probe=next(p for p in probes if (p['dataset'],p['db'])==(dataset,db))
    gallery_pair=[next(g for g in galleries if g['identity_key']==probe['identity_key']),
        next(g for g in galleries if (g['dataset'],g['db'])==(dataset,db)
            and g['connected_group']!=probe['connected_group'])]
    query=original35[probe['relative_path']]
    for gallery in gallery_pair:
        target=original35[gallery['relative_path']]
        if query['status']!='matched' or target['status']!='matched':
            fd_rows.append(dict(dataset=dataset,db=db,status='unmatchable_original_anchors'))
            continue
        matrix=local_matrix(query['feature'],query['mask'],target)
        qa,ga=torch.from_numpy(query['anchors']),torch.from_numpy(target['anchors'])
        plan=freeze_lsar(matrix,qa,ga)
        index=int(plan['weights'].argmax())
        r,c=divmod(index,matrix.shape[1])
        epsilon=.0001
        plus,minus=matrix.clone(),matrix.clone()
        plus[r,c]+=epsilon
        minus[r,c]-=epsilon
        pplan,mplan=freeze_lsar(plus,qa,ga),freeze_lsar(minus,qa,ga)
        stable=torch.equal(plan['weights'],pplan['weights']) and torch.equal(plan['weights'],mplan['weights'])
        numerical=float((pplan['baseline']-mplan['baseline'])/(2*epsilon))
        analytic=float(plan['weights'][r,c])
        error=abs(numerical-analytic)
        fd_rows.append(dict(dataset=dataset,db=db,probe=probe['relative_path'],gallery=gallery['relative_path'],
            status='stable' if stable else 'matching_or_rank_boundary',analytic=analytic,numerical=numerical,
            absolute_error=error,passed=bool(error<=.0003) if stable else None))
fd_frame=pd.DataFrame(fd_rows)
stable_fd=fd_frame[fd_frame.status.eq('stable')]
fd_passed=bool(len(stable_fd)>0 and stable_fd.passed.eq(True).all())
write_csv(FROZEN_DIR/'finite_differences.csv',fd_frame)
print(fd_frame.to_string(index=False))


In [ ]:

dmd_model.to(CHECK_DEVICE)
restorer.to(CHECK_DEVICE)
if CHECK_DEVICE.type=='cuda':
    torch.cuda.reset_peak_memory_stats()
live_rows=[]
for dataset,db in sorted({(p['dataset'],p['db']) for p in probes}):
    probe=next(p for p in probes if (p['dataset'],p['db'])==(dataset,db))
    query=read_cached33(probe,'dry_like','moderate')
    negatives=[g for g in galleries if (g['dataset'],g['db'])==(dataset,db)
        and g['identity_key']!=probe['identity_key'] and g['connected_group']!=probe['connected_group']
        and original35[g['relative_path']]['status']=='matched']
    threshold=source_thresholds[f'{dataset}_{db}']
    if query['status']!='matched' or not negatives or threshold is None:
        live_rows.append(dict(dataset=dataset,db=db,status='blocked_missing_anchor_or_calibration',passed=False))
        continue
    reference=original_train_image(probe)
    degraded,_=simulator.degrade(reference,'dry_like','moderate',
        simulator.case_seed(probe['relative_path'],'dry_like','moderate',SEED))
    patches=all_aligned_patches(degraded,query['anchors'])
    targets=all_aligned_patches(reference,query['anchors'])
    after_feature,after_mask=patch_descriptors(patches,True)
    input_feature,input_mask=patch_descriptors(patches,False)
    feature=after_feature.detach().clone().requires_grad_()
    mask=after_mask.detach().clone().requires_grad_()
    plans,before_values,after_values,input_replayed_values=[],[],[],[]
    for negative in negatives:
        target=original35[negative['relative_path']]
        before_matrix=local_matrix(query['feature'],query['mask'],target)
        plan=freeze_lsar(before_matrix,torch.from_numpy(query['anchors']),torch.from_numpy(target['anchors']))
        after_matrix=local_matrix(feature,mask,target)
        plans.append(plan)
        before_values.append(plan['baseline'].to(CHECK_DEVICE))
        after_values.append(frozen_lsar_score(plan,after_matrix))
        input_replayed_values.append(frozen_lsar_score(plan,local_matrix(input_feature,input_mask,target)))
    before_values,after_values=torch.stack(before_values),torch.stack(after_values)
    input_replayed_values=torch.stack(input_replayed_values)
    input_replay_error=float((input_replayed_values-before_values).abs().max())
    _,input_replay_risk=counterfactual_risks(before_values,input_replayed_values,threshold)
    mean_risk,tail_risk=counterfactual_risks(before_values,after_values,threshold)
    risk_df,risk_dm=torch.autograd.grad(tail_risk,(feature,mask),retain_graph=True)
    raw_df,raw_dm=torch.autograd.grad(after_values.mean(),(feature,mask))
    chain=parameter_vjps(patches,targets,[raw_df.detach(),risk_df.detach()],[raw_dm.detach(),risk_dm.detach()])
    raw,risk=chain['objectives']
    positive=bool(float(tail_risk.detach())>0)
    active=bool(float(tail_risk.detach())>float(input_replay_risk)+5e-6)
    identical=counterfactual_risks(before_values,before_values,threshold)
    row=dict(dataset=dataset,db=db,probe=probe['relative_path'],anchors=len(query['anchors']),
        negative_identities=len(negatives),threshold=threshold,status='checked',
        mean_risk=float(mean_risk.detach()),tail_risk=float(tail_risk.detach()),risk_active=active,
        positive_risk=positive,input_replay_score_error=input_replay_error,input_replay_tail_risk=float(input_replay_risk),
        identity_mean_risk=float(identical[0]),identity_tail_risk=float(identical[1]),
        raw_prediction_gradient_l1=raw['prediction_l1'],raw_parameter_gradient_l1=raw['parameter_l1'],
        tail_parameter_gradient_l1=risk['parameter_l1'],l1=chain['l1_value'],
        l1_parameter_gradient_l2=chain['l1_parameter_l2'],
        auxiliary_to_l1_gradient_ratio=.1*risk['parameter_l2']/chain['l1_parameter_l2']
            if chain['l1_parameter_l2']>0 else None,finite=chain['finite'])
    row['passed']=bool(chain['finite'] and raw['prediction_l1']>0 and raw['parameter_l1']>0
        and row['identity_mean_risk']==0 and row['identity_tail_risk']==0
        and input_replay_error<=1e-6 and (not positive or risk['parameter_l1']>0))
    live_rows.append(row)
    print('Live full-chain check:',json.dumps(row),flush=True)
    del feature,mask,after_feature,after_mask,after_values,before_values,risk_df,risk_dm,raw_df,raw_dm,mean_risk,tail_risk,input_feature,input_mask,input_replayed_values
live_frame=pd.DataFrame(live_rows)
write_csv(FROZEN_DIR/'live_gradients.csv',live_frame)
runtime35=dict(seconds=time.monotonic()-started,check_device=str(CHECK_DEVICE),chunk_size=CHUNK_SIZE,
    peak_allocated_bytes=torch.cuda.max_memory_allocated() if CHECK_DEVICE.type=='cuda' else None,
    peak_reserved_bytes=torch.cuda.max_memory_reserved() if CHECK_DEVICE.type=='cuda' else None)
write_verified(FROZEN_DIR/'runtime.json',json_bytes(runtime35))


In [ ]:

if (state_sha(dmd_model)!=initial_dmd_sha or state_sha(restorer)!=initial_restorer_sha
        or any(p.grad is not None for p in dmd_model.parameters())
        or any(p.grad is not None for p in restorer.parameters())):
    raise RuntimeError('Frozen weights or parameter .grad changed during read-only verification.')
if any(not torch.equal(value,model.state_dict()[key]) for key,value in saved_state.items()):
    raise RuntimeError('Original29 model changed.')
verify_files(MATCHER_DIR,completion['file_sha256'])
if (file_digest(MATCHER_DIR/'completion.json')!=pointer['completion_sha256']
        or json.loads((MATCHER_ROOT/'completed_experiment.json').read_text(encoding='utf-8'))!=pointer):
    raise RuntimeError('Original29 evidence changed.')
verify_files(SOURCE33_DIR,source_completion['file_sha256'])
verify_files(SOURCE34_DIR,source34_completion['file_sha256'])
if (file_digest(SOURCE33_DIR/'completion.json')!=configuration35['source33_completion_sha256']
        or file_digest(SOURCE34_DIR/'completion.json')!=source34_completion_sha):
    raise RuntimeError('33/34 completion changed.')
verify_files(ASSET_ROOT,EXPECTED_DMD_FILES)
verify_files(ORIGINAL_DIR,original_completion['file_sha256'])
verify_files(ORIGINAL_DIR,original_config['source_file_sha256'])
verify_files(original_run,original_run_completion['file_sha256'])
if (file_digest(WEIGHT_PATH)!=EXPECTED_WEIGHT_SHA
        or file_digest(ORIGINAL_DIR/'completion.json')!=original_pointer['completion_sha256']
        or json.loads((SPECIALIZATION_ROOT/'completed_experiment.json').read_text(encoding='utf-8'))!=original_pointer):
    raise RuntimeError('Public DMD/original23 evidence changed.')
live_passed=bool(len(live_frame)>0 and live_frame.passed.eq(True).all())
active_cases=int(live_frame.risk_active.fillna(False).sum()) if 'risk_active' in live_frame else 0
eligible=bool(replay35['passed'] and fd_passed and live_passed and active_cases>0
    and source_completion['quality']['passed'])
result35=dict(status='frozen_before_lsar_check_complete',configuration_sha256=frozen_sha,
    replay_passed=replay35['passed'],finite_difference_passed=fd_passed,full_chain_passed=live_passed,
    active_risk_cases=active_cases,permits_new_matched_pilot=eligible,permits_legacy_30_31=False,
    old34_quality_passed=source34_completion['quality']['passed'],
    scope='ONE explicitly new frozen-before full-template surrogate; no rewrite of34; no independent quality/novelty/restoration claim.')
names=('configuration.json','frozen_lsar.py.txt','calibration_replayed_scores.csv','gate_replayed_scores.csv',
    'plan_metadata.csv','plans.npz','geometry_only_control.csv','replayed_gate_metrics.csv','replay.json',
    'finite_differences.csv','live_gradients.csv','runtime.json')
result35['file_sha256']={name:file_digest(FROZEN_DIR/name) for name in names}
write_verified(FROZEN_DIR/'completion.json',json_bytes(result35))
write_verified(FROZEN_ROOT/'completed_diagnostic.json',json_bytes(dict(directory=str(FROZEN_DIR),
    configuration_sha256=frozen_sha,completion_sha256=file_digest(FROZEN_DIR/'completion.json'))))
print(json.dumps(result35,indent=2))
print(json.dumps(runtime35,indent=2))
print('FROZEN LSA-R FULL-CHAIN CHECK COMPLETE:',FROZEN_DIR)
if eligible:
    print('Technical prerequisite passed for a NEW matched pilot; no training started, no restoration improvement established.')
else:
    print('STOP this loss direction: the ONE authorized frozen-before alternative did not meet its checks. No scalar sweep or gate relaxation.')
print('Legacy30/31 remain blocked;34 remains unchanged.')
